In [0]:
# ==============================================================================
# DATA ASSET MANAGEMENT: RISK & TALENT AUDIT PIPELINE
# AUTO LOADER DEMO: SCHEMA EVOLUTION & RESCUED DATA COLUMN
# ==============================================================================

from pyspark.sql.functions import col, current_timestamp

# --- 1. SETUP ENVIRONMENT & STORAGE PATHS ---
spark.sql("CREATE CATALOG IF NOT EXISTS training_medallion")
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.01_bronze")

base_volume = "/Volumes/training_medallion/00_landing/raw_landing_zone"
risk_landing = f"{base_volume}/risk_management/audit_landing"
risk_checkpoint = f"{base_volume}/meta_checkpoints/risk_audit"
risk_schema = f"{base_volume}/meta_schemas/risk_audit"

# Create landing folder in Unity Catalog Volume
dbutils.fs.mkdirs(risk_landing)
print("Landing directory successfully prepared.")


# --- 2. BATCH 1: BASELINE SCHEMA (TALENT RISK AUDIT) ---
# Initial baseline schema: employee_id (INT), name (STRING), risk_score (INT), audit_status (STRING)
batch1_talent_risk = """{"employee_id": 1001, "name": "Budi Santoso", "risk_score": 15, "audit_status": "LOW"}
{"employee_id": 1002, "name": "Siti Aminah", "risk_score": 78, "audit_status": "HIGH"}"""

dbutils.fs.put(f"{risk_landing}/batch_1_baseline.json", batch1_talent_risk, overwrite=True)

# Auto Loader Ingestion: Batch 1 (Initial Schema Inference)
df_risk_stream1 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", risk_schema)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(risk_landing)
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

q1 = (
    df_risk_stream1.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", risk_checkpoint)
    .option("mergeSchema", "true")
    .table("training_medallion.01_bronze.bronze_talent_risk_audit")
)
q1.awaitTermination()

print("-> Batch 1 ingested. Baseline Schema:")
spark.table("training_medallion.01_bronze.bronze_talent_risk_audit").printSchema()


# --- 3. BATCH 2: SCHEMA EVOLUTION (WITH AUTOMATIC RETRY HANDLING) ---
batch2_evolved = """{"employee_id": 1003, "name": "Rian Pratama", "risk_score": 42, "audit_status": "MEDIUM", "department": "Treasury", "training_completed": true}"""

dbutils.fs.put(f"{risk_landing}/batch_2_new_fields.json", batch2_evolved, overwrite=True)

def run_risk_stream():
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", risk_schema)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(risk_landing)
        .withColumn("ingestion_time", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
    )

    q = (
        df.writeStream
        .format("delta")
        .trigger(availableNow=True)
        .option("checkpointLocation", risk_checkpoint)
        .option("mergeSchema", "true")
        .table("training_medallion.01_bronze.bronze_talent_risk_audit")
    )
    q.awaitTermination()

# Auto Loader throws UNKNOWN_FIELD_EXCEPTION on the 1st run to update schema metadata.
# The 2nd run reads the updated schema and succeeds without data loss.
try:
    print("Running Batch 2 ingestion (Attempt 1)...")
    run_risk_stream()
except Exception as e:
    if "UNKNOWN_FIELD_EXCEPTION" in str(e):
        print("⚡ New columns detected! Auto Loader updated the schema definition. Retrying ingestion...")
        run_risk_stream()
        print("✅ Batch 2 successfully ingested after schema evolution!")
    else:
        raise e

print("\n-> Current Table Schema (reflecting added columns):")
spark.table("training_medallion.01_bronze.bronze_talent_risk_audit").printSchema()


# --- 4. BATCH 3: RESCUED DATA COLUMN (CORRUPTED / TYPE MISMATCH) ---
# Malformed data: 'employee_id' contains a string ("EMP_CORRUPT_99") and invalid risk_score type
batch3_corrupted = """{"employee_id": "EMP_CORRUPT_99", "name": "Dewi Sartika", "risk_score": 90, "audit_status": "CRITICAL", "department": "IT", "training_completed": false}
{"employee_id": 1005, "name": "Unknown Employee", "risk_score": "INVALID_SCORE", "audit_status": "UNKNOWN"}"""

dbutils.fs.put(f"{risk_landing}/batch_3_corrupted.json", batch3_corrupted, overwrite=True)

# Auto Loader isolates malformed values into _rescued_data without stream interruption
df_risk_stream3 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", risk_schema)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(risk_landing)
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

q3 = (
    df_risk_stream3.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", risk_checkpoint)
    .option("mergeSchema", "true")
    .table("training_medallion.01_bronze.bronze_talent_risk_audit")
)
q3.awaitTermination()


# --- 5. DATA ASSET VERIFICATION ---
print("-> Final Bronze Table Inspection (Clean vs Rescued Records):")
display(
    spark.table("training_medallion.01_bronze.bronze_talent_risk_audit")
    .select(
        "employee_id", 
        "name", 
        "risk_score", 
        "audit_status", 
        "department", 
        "training_completed", 
        "_rescued_data"
    )
)

Landing directory successfully prepared.
Wrote 173 bytes.
-> Batch 1 ingested. Baseline Schema:
root
 |-- audit_status: string (nullable = true)
 |-- employee_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- risk_score: long (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- department: string (nullable = true)
 |-- training_completed: boolean (nullable = true)
 |-- audit_id: string (nullable = true)
 |-- dept_code: string (nullable = true)
 |-- flag_reason: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

Wrote 143 bytes.
Running Batch 2 ingestion (Attempt 1)...

-> Current Table Schema (reflecting added columns):
root
 |-- audit_status: string (nullable = true)
 |-- employee_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- risk_score: long (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- in

employee_id,name,risk_score,audit_status,department,training_completed,_rescued_data
null,Dewi Sartika,90,CRITICAL,IT,false,"{""employee_id"":""EMP_CORRUPT_99"",""_file_path"":""/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/audit_landing/batch_3_corrupted.json""}"
1005,Unknown Employee,null,UNKNOWN,null,null,"{""risk_score"":""INVALID_SCORE"",""_file_path"":""/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/audit_landing/batch_3_corrupted.json""}"
101,null,88,null,null,null,null
102,null,25,null,null,null,null
103,null,-5,null,null,null,null
101,null,88,null,null,null,null
1001,null,85,null,null,null,null
1002,null,20,null,null,null,null
1003,Rian Pratama,42,MEDIUM,Treasury,true,null
1001,Budi Santoso,15,LOW,null,null,null
